# Verifiable Quantum Circuit Compilation

## v6 · DFTHC-SOS 7-bit Rounding

**Hypothesis:** choose the sign of each half-way rounding error, then refit the weights, to recover chemistry accuracy with 7-bit angle words. The public `(R, B, C) = (9, 143, 40)` solution uses 8 bits, `J=5`, `T=16,233`, `Q=428`, `lambda_eff≈47.0356`, `error_2≈20.00000 mHa` and `error_1≈13.27 mHa`. Its estimated portfolio score is 23.95. Naive 7-bit attempts failed; notebook 04 found fixed-word error to be dictionary-limited, while notebook 05 supplied the vector gradient.

Even 8-bit words map exactly to `w7=w8/2`. Odd words have two equally distant choices, each `π/128` from the original angle: approximately 48k two-body and 2.8k one-body binary decisions. Words wrap modulo 128, including `255 → ceil(255/2)=128 → 0`.

| Component | 8 bits | 7 bits | Status |
|:--|--:|--:|:--|
| Givens Toffolis | 2,250 | 1,950 | exact scorer formula |
| Angle-loader Toffolis | 9,492 | 8,305.5 | rough `7/8` scaling |
| Persistent angle qubits, J=5 | 40 | 35 | exact `J*brot` |
| J=6 loader workspace | — | at most 75 temporary | current 77 does not fit |

A roughly 10% chemistry improvement could move 23.95 toward 24.7; the scorer-backed table below computes the actual model. This is a CPU chemistry experiment, **not a circuit submission**. It saves best available words even if infeasible. Recompiled loaders and full circuit verification remain necessary.

Attach competition data and `heliosli/qcc-circuits-live` (or `yoyobar/qcc-local-sota-20261031`). Only the public `submission.npz` is extracted. No internet, installs, GPU, or circuit simulation is used. Each search phase has an alarm deadline; the whole research process is killed at 140 minutes, leaving ten minutes below Kaggle's 2.5-hour allowance for shutdown and reporting.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SEED = 20261103
RELEASES = {'heliosli/qcc-circuits-live': 'QCC_RELEASE_MANIFEST.json',
            'yoyobar/qcc-local-sota-20261031': 'portfolio-manifest.json'}
TOTAL_MINUTES = 140
PHASE_MINUTES = dict(naive=15, progressive=40, polish=20, one_body=10, refit=30)
ROUND_FRACTION = .15
ROUND_STEPS = 16                  # last step commits all remaining midpoint words
BCD_SWEEPS = 3
NAIVE_SWEEPS = 10
POLISH_TOP = 64
POLISH_REFIT_EVERY = 8
ERROR_CAP = .0199998
LAMBDA_FLOOR = 49.00001
GAP_FLOOR = 3.9149
REFIT_TRIGGER = .01995
REFERENCE_COST = 3_355_599_588.3155365
PORTFOLIO = {'invsqrt': 20.65035, 'dqi': 64.76241}
PUBLIC_T, PUBLIC_Q = 16233, 428
ANGLE_T8, ANGLE_TEMP, SELECTION = 9492., 77, 11


In [ ]:
# ── Session ───────────────────────────────────────────────────────────────────
import os
# Set before importing NumPy/SciPy; Kaggle CPU allocation is four cores.
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '4'
import sys, glob, json, lzma, time, math, shutil, hashlib, pickle, traceback, signal
import multiprocessing as mp
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize, linear_sum_assignment
try:
    from threadpoolctl import threadpool_limits
    THREAD_LIMIT = threadpool_limits(limits=4)
except ImportError:
    THREAD_LIMIT = None

ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True)
PUBLIC = ROOT / 'rounding_public'; PUBLIC.mkdir(exist_ok=True)
ART = ROOT / 'rounding_phases'; ART.mkdir(exist_ok=True)
RUN_ID = str(time.time_ns())
NB_T0 = time.monotonic()
HARD_END = NB_T0 + TOTAL_MINUTES * 60
np.random.seed(SEED)
pd.set_option('display.max_columns', 30)

def deep_find(name):
    return sorted(Path(p) for p in glob.glob(str(Path('/kaggle/input') / '**' / name), recursive=True))

hits = deep_find('verify_submission_invsqrt.py')
COMP = next((p for p in (Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation'),
                         Path('/kaggle/input/google-verifiable-quantum-circuit-compilation'))
             if (p / 'verifiers').is_dir()), None)
if COMP is None and hits:
    COMP = hits[0].parent.parent.parent
assert COMP is not None, 'Attach the competition data'
release_hits = [(name, deep_find(marker)) for name, marker in RELEASES.items()]
RELEASE_DATASET, found = next(((name, h) for name, h in release_hits if h), (None, []))
assert found, 'Attach heliosli/qcc-circuits-live or yoyobar/qcc-local-sota-20261031'
RELEASE = found[0].parent
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
from dfthcsos import verify as v_dfthcsos
from dfthcsos.scoreinfra import dfthc_math as official_math, reference
from dfthcsos.scoreinfra.costs_prepare import SpinFreeData_V1
print('competition:', COMP, '| release:', RELEASE_DATASET, '| CPU limit: 4', flush=True)

def atomic_pickle(path, value):
    tmp = path.with_suffix('.partial')
    with tmp.open('wb') as f:
        pickle.dump(value, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def phase_snapshot(state):
    # Internal research state only; no pickle is ever loaded from attached inputs.
    atomic_pickle(PHASE_PATH, state)
    complete = state.get('best', state)
    if isinstance(complete, dict) and 'words' in complete:
        save_npz(complete, 'rounding_7bit_best')
        if SUMMARY_PATH.exists():
            summary = json.loads(SUMMARY_PATH.read_text())
            summary['latest_phase_checkpoint'] = small_state(complete)
            summary['latest_phase_state'] = str(PHASE_PATH)
            write_summary(summary)

PHASE_LOG = []
def run_phase(name, function, args, fallback):
    global PHASE_PATH
    PHASE_PATH = ART / (RUN_ID + '_' + name + '.pkl')
    phase_snapshot(fallback)
    end = min(HARD_END - 90, time.monotonic() + 60 * PHASE_MINUTES[name])
    if time.monotonic() >= end:
        PHASE_LOG.append(dict(phase=name, seconds=0, status='skipped: global budget'))
        return fallback
    def timeout_handler(signum, frame):
        raise TimeoutError(name + ' phase deadline')
    start = time.monotonic()
    previous_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.setitimer(signal.ITIMER_REAL, max(.001, end-start))
    status = 'complete'
    try:
        result = function(*args, end=end)
        phase_snapshot(result)
    except TimeoutError:
        status = 'timeout; checkpoint retained'
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, previous_handler)
    PHASE_LOG.append(dict(phase=name, seconds=time.monotonic()-start, status=status))
    with PHASE_PATH.open('rb') as f:
        result = pickle.load(f)
    print(PHASE_LOG[-1], flush=True)
    return result


## Public weights and the exact pair-space target

The manifest hash authenticates the one required file. The reference Hamiltonian and fixed shifts come from `load_inputs`; submitted shifts are not optimization variables. Setup aborts if the packed target disagrees with the official 8-bit evaluator. All heavy cells below define functions; computation starts only in the bounded execution cell.


In [ ]:
def setup():
    global BASE, INPUTS, REF, SHIFT, N, R, B, C, ii, jj, d, L, M, NORM_M2
    global W8, P8, WORDS8, EDGES, HBASE, BASE_NATIVE
    manifest = json.loads((RELEASE / RELEASES[RELEASE_DATASET]).read_text())
    dst = PUBLIC / 'submission.npz'
    if RELEASE_DATASET.startswith('heliosli/'):
        item = next(i for i in manifest['files'] if i['file'] == 'dfthcsos/submission.npz')
        src = RELEASE / item['stored_as']
        if item['stored_as'].endswith('.xz'):
            with lzma.open(src) as fi, dst.open('wb') as fo:
                shutil.copyfileobj(fi, fo, 1 << 22)
        else:
            shutil.copyfile(src, dst)
        assert hashlib.sha256(dst.read_bytes()).hexdigest() == item['sha256']
    else:
        shutil.copyfile(RELEASE / 'submission/dfthcsos/submission.npz', dst)
    with np.load(dst, allow_pickle=False) as z:
        BASE = {k: np.array(z[k]) for k in z.files}
    INPUTS = v_dfthcsos.load_inputs(dst)
    REF, SHIFT = INPUTS.ref, INPUTS.S
    N = int(REF['orbitals']); R, B, C = BASE['W'].shape
    W8 = BASE['W'].astype(float); P8 = BASE['H2_I'] - W8.sum(1)
    WORDS8 = BASE['angle_words'].astype(np.int64); EDGES = BASE['tree_edges']
    assert (N, R, B, C) == (76, 9, 143, 40), 'Cost calibration is for the public shape'
    assert WORDS8.shape == (N+R*B, N-1) and np.all((WORDS8 >= 0) & (WORDS8 < 256))
    if 'brot' in BASE:
        assert int(BASE['brot']) == 8
    ii, jj = np.tril_indices(N); d = np.sqrt(np.where(ii == jj, 1., 2.)); L = len(ii)
    M = REF['tei'][ii[:, None], jj[:, None], ii[None, :], jj[None, :]].copy()
    diagonal = (ii == jj).astype(float); shifted = SHIFT[ii, jj]
    M += .5 * (shifted[:, None]*diagonal[None, :] + diagonal[:, None]*shifted[None, :])
    M *= d[:, None]*d[None, :]; M = np.ascontiguousarray(.5*(M+M.T))
    NORM_M2 = float(np.sum(M*M))
    HBASE = (REF['oei'] - .5*np.einsum('prrq->pq', REF['tei'])
             - .5*int(REF['electrons'])*SHIFT + INPUTS.beta_1*np.eye(N))
    t = time.monotonic(); BASE_NATIVE = native(WORDS8, W8, P8, brot=8)
    pair_error = error_of(copies(projectors(decode(WORDS8, 8)[N:].reshape(R, B, N)), W8))
    assert abs(pair_error-BASE_NATIVE['error_2']) < 1e-7, 'target rebuild disagrees with official evaluator'
    assert v_dfthcsos.passes_accuracy(BASE_NATIVE), 'public chemistry baseline failed'
    print('8-bit native metrics:', BASE_NATIVE, flush=True)
    print('Pair-space parity:', pair_error, '| official evaluator seconds:', time.monotonic()-t, flush=True)
    self_checks()

def decode(words, brot=7):
    U = official_math.reconstruct_u_vectors(np.asarray(words)/2**(brot-1), EDGES)
    return U / np.linalg.norm(U, axis=1, keepdims=True)

def native(words, w, p, brot=7):
    return official_math.evaluate_dfthc_angle_words(
        np.asarray(words, dtype=np.int64), w, p, SHIFT, INPUTS.beta_1, REF,
        brot=brot, tree_edges=EDGES, energy_upper=reference.UPPER_ENERGY)

def projectors(U):
    return U[..., ii]*U[..., jj]*d

def copies(P, W):
    return np.ascontiguousarray((W.transpose(0, 2, 1) @ P).transpose(2, 0, 1)).reshape(L, R*C)

def error_of(F):
    G = F.T @ F
    return math.sqrt(max(NORM_M2 - 2*float(np.sum(F*(M@F))) + float(np.sum(G*G)), 0.))

def unpack(g):
    S = np.zeros(g.shape[:-1]+(N, N)); val = g/d
    S[..., ii, jj] = val; S[..., jj, ii] = val
    return S

def vector_gradient(U, W):
    P = projectors(U); F = copies(P, W)
    GF = (4*(F@(F.T@F)-M@F)).reshape(L, R, C).transpose(1, 0, 2)
    gP = W @ GF.transpose(0, 2, 1)
    gU = 2*(unpack(gP) @ U[..., None])[..., 0]
    return gU - U*np.sum(U*gU, axis=-1, keepdims=True)

def refit_w(U, W, sweeps, end):
    """Exact rank-C block minimizers; finite sweeps give a local BCD fit, not a global optimum."""
    P = projectors(U)
    Q = [np.linalg.qr(P[r].T)[0] for r in range(R)]
    QMQ = [q.T @ (M@q) for q in Q]
    Fr = [P[r].T @ W[r] for r in range(R)]
    previous = error_of(np.concatenate(Fr, axis=1))
    for sweep in range(sweeps):
        old_W = W.copy()
        for r in range(R):
            if time.monotonic() >= end-5:
                return W
            K = QMQ[r].copy()
            for s in range(R):
                if s != r:
                    t = Q[r].T @ Fr[s]; K -= t@t.T
            val, vec = np.linalg.eigh(.5*(K+K.T)); top = np.argsort(val)[::-1][:C]
            new = (Q[r] @ vec[:, top])*np.sqrt(np.maximum(val[top], 0.))
            # Preserve the old copy gauge as closely as possible while keeping p fixed.
            left, _, right = np.linalg.svd(new.T@Fr[r], full_matrices=False)
            new = new @ (left@right)
            wr = np.linalg.lstsq(P[r].T, new, rcond=None)[0]
            actual = P[r].T@wr
            if np.linalg.norm(actual-new) <= 1e-8*max(1., np.linalg.norm(new)):
                W = W.copy(); W[r] = wr; Fr[r] = actual
        current = error_of(np.concatenate(Fr, axis=1))
        if current > previous+1e-10:
            return old_W
        improvement = previous-current; previous = current
        print('BCD sweep', sweep+1, 'error_2 mHa', 1e3*previous, flush=True)
        if improvement < 1e-9:
            break
    return W


## Cost of seven-bit words

`SpinFreeData_V1` supplies exact preparation, Givens and heuristic qubits. Angle Toffolis use rough `brot/8` scaling of the public 9,492. The shared-workspace formula and `J*brot` persistence are exact conditional on the stated temporary workspace. `J=6` is shown as blocked unless `11+6*7+angle_temp<=128`; it requires at most 75 temporary qubits versus today's 77. No J-dependent Toffoli gain is invented.


In [ ]:
def cost_table(lam):
    rows = []
    for brot in (8, 7):
        for J in (5, 6):
            required_temp = 128-SELECTION-J*brot
            fits = SELECTION+J*brot+ANGLE_TEMP <= 128
            model = SpinFreeData_V1(R=R, B=B, C=C, N=N, chi=10, beta=brot)
            oq, iq = model.outer_state_prep_qubit_cost(), model.inner_state_prep_qubit_cost()
            outer, inner, givens = model.outer_state_prep(), model.inner_state_prep(), model.givens_rotations()
            angle = ANGLE_T8*brot/8
            T = outer+inner+givens+angle
            Q = oq['persistent']+iq['persistent']+J*brot+max(oq['temporary'], iq['temporary'], ANGLE_TEMP)
            cost = lam*T*Q; ratio = REFERENCE_COST/cost
            rows.append(dict(brot=brot, J=J, outer_T=outer, inner_T=inner, givens_T=givens,
                angle_T_rough=angle, angle_persistent=J*brot, angle_temp=ANGLE_TEMP,
                required_temp_max=required_temp, loader_fits=fits,
                heuristic_persistent=oq['persistent']+iq['persistent'],
                heuristic_temporary=max(oq['temporary'], iq['temporary']),
                T_model=T, Q_model=Q, cost_model=cost if fits else None,
                chemistry_ratio=ratio if fits else None,
                LB_model=(ratio*PORTFOLIO['invsqrt']*PORTFOLIO['dqi'])**(1/3) if fits else None,
                status='modeled; loaders uncompiled' if fits else 'blocked by 128-qubit cap'))
    assert (rows[0]['T_model'], rows[0]['Q_model']) == (PUBLIC_T, PUBLIC_Q)
    print(pd.DataFrame(rows).to_string(index=False), flush=True)
    return rows


## Naive rounding controls

Floor and round-half-up are separate controls; NumPy's ties-to-even rounding is intentionally avoided. Each control is evaluated first with public `W,p`, then with BCD-refitted `W` and the **same public** `p=H2_I-W.sum(1)`. The corresponding saved `H2_I` is always reconstructed from that `p` and the candidate `W`. BCD minimizes the two-body error; it can change the one-body target even with the copy-gauge alignment.


In [ ]:
def record(words, W, p, label, end=None):
    e2 = error_of(copies(projectors(decode(words)[N:].reshape(R, B, N)), W))
    metrics = native(words, W, p)
    assert abs(e2-metrics['error_2']) < 1e-7, (label, e2, metrics['error_2'])
    out = dict(label=label, words=words.copy(), W=W.copy(), p=p.copy(), pair_error_2=e2, metrics=metrics)
    print(label, {k: metrics[k] for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')}, flush=True)
    return out

def naive_search(end):
    controls = []
    for label, words in (('floor', WORDS8//2), ('half_up', ((WORDS8+1)//2)%128)):
        controls.append(record(words, W8, P8, label+'_public_W'))
        best = min(controls, key=lambda s:s['pair_error_2'])
        phase_snapshot(dict(best=best, controls=controls.copy()))
    raw_controls = controls.copy()
    for index, raw in enumerate(raw_controls):
        words = raw['words']
        local_end = min(end, time.monotonic()+max(0., end-time.monotonic())/(len(raw_controls)-index))
        W = refit_w(decode(words)[N:].reshape(R, B, N), W8.copy(), NAIVE_SWEEPS, local_end)
        controls.append(record(words, W, P8, raw['label'].replace('_public_W', '_BCD_W')))
        best = min(controls, key=lambda s:s['pair_error_2'])
        phase_snapshot(dict(best=best, controls=controls.copy()))
    return dict(best=best, controls=controls)


## Progressive gradient rounding

The starting two-body angles are the exact 8-bit midpoints. Each round commits the largest-gradient 15% of the remaining odd words against the gradient sign, then refits `W`. The final round, including an early final round near the deadline, commits all remaining words. We use analytic reverse differentiation of the **same ordered Givens rotations** as the official decoder, vectorized over every row; no per-word finite-difference loop is needed. The normalized-vector gradient comes from notebook 05.

Only fully discrete states compete with the naive baselines. If a hard phase timeout interrupts a partial round, the last complete 7-bit checkpoint survives. A small in-notebook finite-difference check validates both the angle chain rule and wrap-around before search.


In [ ]:
def angle_vjp(theta, grad_unit):
    """d f/d theta (radians), batched reverse pass through the official Givens tree."""
    nrows = len(theta); v = np.zeros((nrows, N)); v[:, 0] = 1.
    before = []
    for k, (a, b) in enumerate(EDGES):
        x, y = v[:, a].copy(), v[:, b].copy(); before.append((x, y))
        co, si = np.cos(theta[:, k]), np.sin(theta[:, k])
        v[:, a], v[:, b] = co*x-si*y, si*x+co*y
    norm = np.linalg.norm(v, axis=1, keepdims=True); u = v/norm
    adj = (grad_unit-u*np.sum(u*grad_unit, axis=1, keepdims=True))/norm
    result = np.zeros_like(theta)
    for k in range(N-2, -1, -1):
        a, b = EDGES[k]; x, y = before[k]
        co, si = np.cos(theta[:, k]), np.sin(theta[:, k])
        ga, gb = adj[:, a].copy(), adj[:, b].copy()
        result[:, k] = ga*(-si*x-co*y)+gb*(co*x-si*y)
        adj[:, a], adj[:, b] = co*ga+si*gb, -si*ga+co*gb
    return result

def self_checks():
    rng = np.random.default_rng(SEED)
    theta = rng.uniform(0, 2*np.pi, (3, N-1)); g = rng.normal(size=(3, N))
    direction = rng.normal(size=theta.shape); direction /= np.linalg.norm(direction)
    h = 1e-6
    plus = decode((theta+h*direction)*64/np.pi)
    minus = decode((theta-h*direction)*64/np.pi)
    fd = float(np.sum((plus-minus)*g)/(2*h))
    an = float(np.sum(angle_vjp(theta, g)*direction))
    assert abs(fd-an) < 1e-6*max(1., abs(fd)), ('angle VJP', fd, an)
    base = WORDS8[:3].copy(); even = base%2 == 0
    assert np.all((base//2)[even]*2 == base[even])
    zero = np.zeros((1, N-1)); wrap = zero.copy(); wrap[0, 0] = 128
    assert np.allclose(decode(zero), decode(wrap), atol=1e-12)
    U = decode(WORDS8, 8)[N:].reshape(R, B, N)
    du = rng.normal(size=U.shape); du /= np.linalg.norm(du)
    def f(V):
        V = V/np.linalg.norm(V, axis=-1, keepdims=True)
        return error_of(copies(projectors(V), W8))**2
    h = 1e-4
    fd = (f(U+h*du)-f(U-h*du))/(2*h)
    an = float(np.sum(vector_gradient(U, W8)*du))
    assert abs(fd-an) < 2e-4*max(1., abs(fd)), ('vector gradient', fd, an)
    print('Angle VJP, vector gradient, even-word mapping and periodic wrap self-checks PASS', flush=True)

def progressive_search(seed_state, end):
    best = seed_state; W = W8.copy()
    theta = WORDS8[N:]*np.pi/128
    odd = WORDS8[N:]%2 == 1; pending = odd.copy()
    low = WORDS8[N:]//2
    history = []; last_step_seconds = 0.
    for step in range(ROUND_STEPS):
        step_start = time.monotonic()
        if step_start >= end-60:
            break
        U = decode(theta*64/np.pi).reshape(R, B, N)
        grad = angle_vjp(theta, vector_gradient(U, W).reshape(-1, N))
        ids = np.flatnonzero(pending)
        if not len(ids):
            break
        finish_now = step == ROUND_STEPS-1 or end-time.monotonic() < max(120., 1.5*last_step_seconds+60.)
        count = len(ids) if finish_now else max(1, math.ceil(ROUND_FRACTION*len(ids)))
        selected = ids[np.argsort(np.abs(grad.ravel()[ids]))[-count:]]
        upper = grad.ravel()[selected] < 0
        theta.ravel()[selected] = (low.ravel()[selected]+upper)*np.pi/64
        pending.ravel()[selected] = False
        U = decode(theta*64/np.pi).reshape(R, B, N)
        W = refit_w(U, W, BCD_SWEEPS, end)
        e2 = error_of(copies(projectors(U), W))
        history.append(dict(step=step+1, undecided=int(pending.sum()), error_2_mHa=1e3*e2))
        print('progressive', history[-1], flush=True)
        last_step_seconds = time.monotonic()-step_start
        if not pending.any():
            words = seed_state['words'].copy()
            words[N:] = np.rint(theta*64/np.pi).astype(np.int64)%128
            assert np.all((words[N:]==low) | (words[N:]==(low+odd)%128))
            candidate = record(words, W, P8, 'progressive')
            candidate['history'] = history
            if candidate['pair_error_2'] < best['pair_error_2']:
                best = candidate
            phase_snapshot(best)
    return best


## Exact single-word polish

First-order gains only rank candidates. Each proposed two-body flip changes a single projector, so the exact squared-error delta is a rank-two residual update. We accept only negative exact deltas, update the residual, and periodically run BCD. This avoids reconstructing the full fourth-order tensor for each trial. Every accepted word remains one of its original floor/ceil choices, including wrap-around. Checkpoints keep the best complete discrete dictionary.


In [ ]:
def flip_delta(D, Fgroup, wrow, dv):
    a = Fgroup@wrow; s = float(wrow@wrow); z = D@dv
    vv = float(dv@dv); av = float(a@dv)
    change = (4*float(a@z)+2*s*float(dv@z)
              +2*float(a@a)*vv+2*av*av+4*s*av*vv+s*s*vv*vv)
    return change, a, s

def polish_search(seed_state, end):
    words = seed_state['words'].copy(); W = seed_state['W'].copy(); best = seed_state
    low = WORDS8[N:]//2; high = (low+1)%128; odd = WORDS8[N:]%2 == 1
    accepted = 0; rounds = 0
    while time.monotonic() < end-40:
        U = decode(words)[N:].reshape(R, B, N); P = projectors(U); F = copies(P, W)
        D = F@F.T-M; sq = float(np.sum(D*D))
        grad = angle_vjp(words[N:]*np.pi/64, vector_gradient(U, W).reshape(-1, N))
        alternate = np.where(words[N:]==low, high, low)
        step = np.where(words[N:]==low, np.pi/64, -np.pi/64)
        predicted = np.where(odd, grad*step, np.inf)
        ids = np.argsort(predicted.ravel())[:min(POLISH_TOP, int(odd.sum()))]
        improved = False
        for idx in ids:
            if time.monotonic() >= end-20:
                break
            row, angle = np.unravel_index(idx, odd.shape); r, b = divmod(row, B)
            proposal = words[N+row:N+row+1].copy(); proposal[0, angle] = alternate[row, angle]
            unew = decode(proposal)[0]; pnew = projectors(unew)
            dv = pnew-P[r, b]; Fg = F[:, r*C:(r+1)*C]
            change, a, s = flip_delta(D, Fg, W[r, b], dv)
            if change < -1e-13:
                # Validate the rank-two identity once against a direct pair-space residual.
                if accepted == 0:
                    checkF = F.copy(); checkF[:, r*C:(r+1)*C] += dv[:, None]*W[r, b][None, :]
                    direct = float(np.sum((checkF@checkF.T-M)**2))-sq
                    assert abs(direct-change) < 1e-8, ('flip delta', direct, change)
                D += np.outer(a, dv)+np.outer(dv, a)+s*np.outer(dv, dv)
                Fg += dv[:, None]*W[r, b][None, :]
                words[N+row, angle] = proposal[0, angle]; P[r, b] = pnew
                sq += change; accepted += 1; improved = True
                if accepted % POLISH_REFIT_EVERY == 0:
                    break
        rounds += 1
        if improved:
            W = refit_w(decode(words)[N:].reshape(R, B, N), W, BCD_SWEEPS, end)
            current = record(words, W, P8, 'smart_polish')
            current.update(flips=accepted, rounds=rounds)
            if current['pair_error_2'] < best['pair_error_2']:
                best = current; phase_snapshot(best)
        print('polish round', rounds, 'accepted', accepted, 'best mHa', 1e3*best['pair_error_2'], flush=True)
        if not improved:
            break
    return best


## One-body rounding with the exact error

For fixed two-body `W,p`, `h1_prime` and its eigenpairs are fixed. We cache them and reproduce the verifier's maximum-overlap assignment exactly, recomputing that assignment for every flip. This makes each exact one-body evaluation a small 76-orbital operation. Its runtime and parity are measured against a full official call. Floor, half-up and the current rows compete before a time-boxed greedy pass; the full native metrics are checked again afterward. The public ≈6.7 mHa slack need not survive a two-body BCD refit.


In [ ]:
def one_body_context(words, W, p):
    U = decode(words)[N:].reshape(R, B, N)
    F = copies(projectors(U), W).reshape(L, R, C)
    packed = -np.einsum('rc,lrc->l', p, F)/d
    h = HBASE.copy(); h[ii, jj] += packed; h[jj, ii] = h[ii, jj]
    eig, vec = np.linalg.eigh(h)
    return h, eig, vec

def one_error(rows, context, gradient=False):
    h, eig, vec = context; u = decode(rows)
    a, b = linear_sum_assignment(-np.abs(u@vec))
    matched = np.empty(N, dtype=int); matched[a] = b
    coeff = eig[matched]; residual = u.T@(coeff[:, None]*u)-h
    sq = float(np.sum(residual*residual))
    if gradient:
        gu = 4*coeff[:, None]*(u@residual)
        return sq, angle_vjp(rows*np.pi/64, gu)
    return math.sqrt(sq)

def one_body_search(seed_state, end):
    words = seed_state['words'].copy(); W, p = seed_state['W'], seed_state['p']
    context = one_body_context(words, W, p)
    t = time.monotonic(); official = native(words, W, p); official_s = time.monotonic()-t
    t = time.monotonic(); e = one_error(words[:N], context); fast_s = time.monotonic()-t
    assert abs(e-official['error_1']) < 2e-8, ('one-body parity', e, official['error_1'])
    print('one-body timing: cached', fast_s, 's; full official', official_s, 's; cached exact evaluator selected', flush=True)
    options = [words[:N].copy(), WORDS8[:N]//2, ((WORDS8[:N]+1)//2)%128]
    rows = min(options, key=lambda q:one_error(q, context)).copy()
    low = WORDS8[:N]//2; high = (low+1)%128; odd = WORDS8[:N]%2 == 1
    words[:N] = rows
    best = record(words, W, p, 'smart_one_body'); phase_snapshot(best)
    accepted = 0
    while time.monotonic() < end-20:
        sq, grad = one_error(rows, context, gradient=True)
        alt = np.where(rows==low, high, low)
        step = np.where(rows==low, np.pi/64, -np.pi/64)
        ids = np.argsort(np.where(odd, grad*step, np.inf).ravel())[:min(256, int(odd.sum()))]
        improved = False
        for idx in ids:
            if time.monotonic() >= end-15:
                break
            trial = rows.copy(); trial.ravel()[idx] = alt.ravel()[idx]
            val = one_error(trial, context)**2
            if val < sq-1e-14:
                rows = trial; sq = val; accepted += 1; improved = True
                break
        if not improved:
            break
        words[:N] = rows
        # Saving the words needs no expensive official tensor evaluation each time.
        current = dict(best, words=words.copy(), one_body_flips=accepted,
                       metrics=dict(best['metrics'], error_1=math.sqrt(sq)))
        best = current; phase_snapshot(best)
        if accepted%20 == 0:
            print('one-body accepted', accepted, 'error_1 mHa', 1e3*math.sqrt(sq), flush=True)
    final = record(words, W, p, 'smart_one_body')
    final['one_body_flips'] = accepted
    return final


## Fixed-word constrained refit

If **both** errors are at most 19.95 mHa, reuse notebook 02's augmented-Lagrangian L-BFGS refit of `W,p`. The safety targets are `error_1,error_2≤19.9998 mHa`, `Lambda≥49.00001` and `E_gap≥3.9149`. Analytic fixed-word metrics must match the official 7-bit evaluator before optimization; directional derivatives are checked away from absolute-value cusps. Only officially checked feasible points can replace a feasible checkpoint.

The pair-space error and gradient use Gram products rather than forming the full residual for every L-BFGS evaluation. A 30-minute alarm preserves the last valid checkpoint; a native call may delay alarm handling, while the outer watchdog enforces the global limit.


In [ ]:
class FixedWordModel:
    def __init__(self):
        self.ii, self.jj = np.tril_indices(N)
        self.d = np.sqrt(np.where(self.ii == self.jj, 1., 2.))
        self.projectors = BODY[:, :, self.ii] * BODY[:, :, self.jj] * self.d
        target = REF['tei'][self.ii[:, None], self.jj[:, None], self.ii[None, :], self.jj[None, :]].copy()
        diagonal = (self.ii == self.jj).astype(float)
        shifted = SHIFT[self.ii, self.jj]
        target += .5 * (shifted[:, None]*diagonal[None, :] + diagonal[:, None]*shifted[None, :])
        self.target = target * self.d[:, None] * self.d[None, :]
        self.hbase = (REF['oei'] - .5*np.einsum('prrq->pq', REF['tei'])
                      - .5*int(REF['electrons'])*SHIFT + INPUTS.beta_1*np.eye(N))
        self.constant = float(reference.UPPER_ENERGY) - float(REF['ecore']) + int(REF['electrons'])*INPUTS.beta_1

    def pack(self, w, p):
        return np.r_[w.ravel(), p.ravel()]

    def unpack(self, z, shape):
        size = int(np.prod(shape))
        return z[:size].reshape(shape), z[size:].reshape(shape[::2])

    def evaluate(self, z, shape):
        w, p = self.unpack(z, shape)
        a = np.einsum('rbc,rbl->rcl', w, self.projectors, optimize=True)
        flat = a.reshape(-1, len(self.d))
        gram = flat @ flat.T
        flatM = flat @ self.target
        e2sq = max(NORM_M2 - 2*float(np.sum(flat*flatM)) + float(np.sum(gram*gram)), 0.)
        packed_h = -np.einsum('rc,rcl->l', p, a) / self.d
        h = self.hbase.copy()
        h[self.ii, self.jj] += packed_h
        h[self.jj, self.ii] = h[self.ii, self.jj]
        eig, vec = np.linalg.eigh(h)
        rows, cols = linear_sum_assignment(-np.abs(VECTORS[:N] @ vec))
        matched = np.empty(N, dtype=int); matched[rows] = cols
        decoded = VECTORS[:N].T @ (eig[matched, None] * VECTORS[:N])
        residual = decoded - h
        e1sq = float(np.sum(residual*residual))
        h2i = p + w.sum(axis=1)
        norm = np.abs(w).sum(axis=1) + np.abs(h2i)
        lam = float(np.abs(eig).sum() + .25*np.sum(norm*norm))
        gap = float(self.constant + np.abs(eig).sum() - eig.sum() + .5*np.sum(p*p))
        discriminant = gap*(2*lam-gap)
        if discriminant <= 0:
            raise ValueError('Left the physical lambda_eff domain')
        eff = math.sqrt(discriminant)
        signh = (vec*np.sign(eig)) @ vec.T
        row_coeff = np.einsum('ij,jk,ik->i', VECTORS[:N], residual, VECTORS[:N], optimize=True)
        eig_coeff = np.empty(N); eig_coeff[matched] = row_coeff
        errorh = 2*((vec*eig_coeff) @ vec.T - residual)

        def through_h(g):
            orbital = np.einsum('rbp,pq,rbq->rb', BODY, g, BODY, optimize=True)
            gw = -orbital[:, :, None]*p[:, None, :]
            gp = -np.einsum('rcl,l->rc', a, g[self.ii, self.jj]*self.d, optimize=True)
            return gw, gp

        lw, lp = through_h(signh)
        lw += .5*norm[:, None, :]*(np.sign(w)+np.sign(h2i)[:, None, :])
        lp += .5*norm*np.sign(h2i)
        gw, gp = through_h(signh-np.eye(N)); gp += p
        e1w, e1p = through_h(errorh)
        ga = (4*(gram@flat-flatM)).reshape(a.shape)
        e2w = np.einsum('rcl,rbl->rbc', ga, self.projectors, optimize=True)
        gradlam = self.pack(lw, lp)
        gradgap = self.pack(gw, gp)
        gradients = dict(lambda_eff=(gap*gradlam+(lam-gap)*gradgap)/eff,
                         Lambda=gradlam, E_gap=gradgap,
                         error_1_sq=self.pack(e1w, e1p),
                         error_2_sq=self.pack(e2w, np.zeros_like(p)))
        metrics = dict(lambda_eff=eff, Lambda=lam, E_gap=gap,
                       error_1_sq=e1sq, error_2_sq=e2sq,
                       error_1=math.sqrt(e1sq), error_2=math.sqrt(e2sq))
        return metrics, gradients

    def constraints(self, m, g):
        caps = np.array([ERROR_CAP**2, ERROR_CAP**2, GAP_FLOOR, LAMBDA_FLOOR])
        c = np.array([m['error_1_sq']-caps[0], m['error_2_sq']-caps[1],
                      caps[2]-m['E_gap'], caps[3]-m['Lambda']]) / caps
        jac = np.stack([g['error_1_sq']/caps[0], g['error_2_sq']/caps[1],
                        -g['E_gap']/caps[2], -g['Lambda']/caps[3]])
        return c, jac


def check_model(model, w, p, seed=SEED):
    rng = np.random.default_rng(seed)
    z = model.pack(w, p)
    # Nonzero perturbations avoid almost all L1 cusps in the public checkpoint.
    z += rng.normal(size=z.size)*1e-7
    ww, pp = model.unpack(z, w.shape)
    got, grads = model.evaluate(z, w.shape)
    official = native_metrics(ww, pp)
    for k in ('lambda_eff', 'Lambda', 'E_gap', 'error_1', 'error_2'):
        assert abs(got[k]-official[k]) < 2e-8, (k, got[k], official[k])
    # Use coordinates well inside their L1 cells to avoid cancellation from tiny finite steps.
    direction = rng.normal(size=z.size)
    dw, dp = model.unpack(direction, w.shape)
    interior = np.abs(ww.sum(1)+pp) > 1e-3
    dw *= np.abs(ww) > 1e-4
    # At offset cusps keep H2_I constant while still testing coupled W/p derivatives.
    dp[:] = np.where(interior, dp, -dw.sum(axis=1))
    assert np.linalg.norm(direction) > 0
    direction /= np.linalg.norm(direction)
    h = 1e-5
    wp, pp_plus = model.unpack(z+h*direction, w.shape)
    wm, pp_minus = model.unpack(z-h*direction, w.shape)
    assert np.array_equal(np.sign(wp), np.sign(wm))
    assert np.array_equal(np.sign(pp_plus+wp.sum(1)), np.sign(pp_minus+wm.sum(1)))
    plus, _ = model.evaluate(z+h*direction, w.shape)
    minus, _ = model.evaluate(z-h*direction, w.shape)
    for k in ('lambda_eff', 'Lambda', 'E_gap', 'error_1_sq', 'error_2_sq'):
        finite = (plus[k]-minus[k])/(2*h)
        analytic = float(grads[k] @ direction)
        assert abs(finite-analytic) < 2e-4*max(1., abs(analytic)), (k, finite, analytic, h)
    print('Native parity and joint directional derivatives PASS', w.shape, flush=True)


In [ ]:
def eligible(m):
    return (all(np.isfinite(m[k]) for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff'))
            and m['error_1'] <= ERROR_CAP and m['error_2'] <= ERROR_CAP
            and m['Lambda'] >= LAMBDA_FLOOR and m['E_gap'] >= GAP_FLOOR
            and v_dfthcsos.passes_accuracy(m))

def native_metrics(w, p):
    return native(REFIT_WORDS, w, p)

def constrained_refit(seed_state, end):
    global REFIT_WORDS, VECTORS, BODY
    REFIT_WORDS = seed_state['words'].copy(); VECTORS = decode(REFIT_WORDS)
    BODY = VECTORS[N:].reshape(R, B, N)
    model = FixedWordModel(); shape = seed_state['W'].shape
    check_model(model, seed_state['W'], seed_state['p'])
    z = model.pack(seed_state['W'], seed_state['p']); dual = np.zeros(4); rho = 10.
    best = seed_state; best_eff = float('inf'); iterations = 0
    if eligible(seed_state['metrics']):
        best_eff = seed_state['metrics']['lambda_eff']
    def checkpoint(point):
        nonlocal best, best_eff, iterations
        if time.monotonic() >= end-15:
            raise TimeoutError('refit budget')
        iterations += 1
        m, _ = model.evaluate(point, shape)
        if (m['error_1'] <= ERROR_CAP and m['error_2'] <= ERROR_CAP
                and m['Lambda'] >= LAMBDA_FLOOR and m['E_gap'] >= GAP_FLOOR
                and m['lambda_eff'] < best_eff):
            w, p = model.unpack(point, shape)
            official = native_metrics(w, p)
            if eligible(official):
                best = dict(seed_state, W=w.copy(), p=p.copy(), metrics=official,
                            pair_error_2=official['error_2'], label='constrained_refit')
                best_eff = official['lambda_eff']; phase_snapshot(best)
        if iterations%20 == 0:
            print('refit iteration', iterations, 'lambda_eff', m['lambda_eff'],
                  'errors mHa', 1e3*m['error_1'], 1e3*m['error_2'], flush=True)
    try:
        checkpoint(z)
        for outer in range(8):
            def objective(point):
                if time.monotonic() >= end-15:
                    raise TimeoutError('refit budget')
                try:
                    m, g = model.evaluate(point, shape)
                except ValueError as exc:
                    if 'physical lambda_eff domain' not in str(exc):
                        raise
                    return float('inf'), np.zeros_like(point)
                c, jac = model.constraints(m, g)
                positive = np.maximum(0., dual+rho*c)
                value = math.log(m['lambda_eff'])+np.sum(positive**2-dual**2)/(2*rho)
                grad = g['lambda_eff']/m['lambda_eff']+positive@jac
                return value, grad
            fit = minimize(objective, z, method='L-BFGS-B', jac=True, callback=checkpoint,
                           options=dict(maxiter=250, maxls=30, maxcor=12, ftol=1e-14, gtol=1e-9))
            z = fit.x; checkpoint(z)
            m, g = model.evaluate(z, shape); c, _ = model.constraints(m, g)
            dual = np.maximum(0., dual+rho*c); rho = min(rho*10, 1e8)
            print('refit outer', outer, 'constraints', c, str(fit.message), flush=True)
    except TimeoutError:
        print('Refit deadline; preserving best native-checked point', flush=True)
    return best


## Bounded execution and checkpoints

The setup and search execute in one isolated Linux process group. Each search phase has an alarm deadline and atomic snapshots, without forking after BLAS work. The outer process watchdog also covers native calls that delay alarm handling. The outer watchdog enforces the 140-minute session limit, including input loading, target construction, parity checks and final official evaluation. A timeout retains complete discrete words; a failed parity assertion aborts rather than continuing with an incorrect target.

`rounding_7bit_best.npz` always contains the best discrete chemistry checkpoint. `rounding_7bit_candidate.npz` is written only after official accuracy passes. These NPZs carry `brot=7` as research metadata; the full verifier derives precision from actual angle circuits, so this does not convert existing 8-bit loaders.


In [ ]:
SUMMARY_PATH = ROOT / 'rounding_7bit.json'

def clean_json(value):
    if isinstance(value, dict):
        return {str(k):clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if math.isfinite(float(value)) else None
    return value

def write_summary(data):
    data = clean_json(data)
    tmp = SUMMARY_PATH.with_suffix('.partial.json')
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False)); os.replace(tmp, SUMMARY_PATH)

def save_npz(state, name):
    words, W, p = state['words'], state['W'], state['p']
    assert words.dtype.kind in 'iu' and words.shape == WORDS8.shape
    assert np.all((words >= 0)&(words < 128))
    low, high = WORDS8//2, ((WORDS8+1)//2)%128
    assert np.all((words == low)|(words == high)), 'a word escaped its floor/ceil choices'
    assert np.all(np.isfinite(W)) and np.all(np.isfinite(p))
    data = dict(BASE, angle_words=words.astype(np.int64), brot=np.array(7), W=W,
                H2_I=p+W.sum(1), U=decode(words)[N:].reshape(R, B, N))
    if 'offset_W' in data:
        data['offset_W'] = p.copy()
    tmp = ROOT / (name+'.partial.npz'); dst = ROOT / (name+'.npz')
    np.savez(tmp, **data)
    with np.load(tmp, allow_pickle=False) as z:
        assert np.array_equal(z['angle_words'], words) and int(z['brot']) == 7
        assert np.allclose(z['H2_I']-z['W'].sum(1), p, rtol=0, atol=1e-12)
    os.replace(tmp, dst)
    return str(dst)

def small_state(state):
    if state is None:
        return None
    m = state['metrics']
    return dict(label=state['label'], pair_error_2_mHa=1e3*state['pair_error_2'],
                error_1_mHa=1e3*m['error_1'], error_2_mHa=1e3*m['error_2'],
                gap_to_20_mHa=dict(error_1=1e3*m['error_1']-20, error_2=1e3*m['error_2']-20),
                metrics=m)

def publish(state, controls, costs, refit_ran=False, status='running'):
    best_path = save_npz(state, 'rounding_7bit_best')
    m = state['metrics']; feasible = bool(v_dfthcsos.passes_accuracy(m))
    candidate_path = save_npz(state, 'rounding_7bit_candidate') if feasible else None
    gap = max(m['error_1'], m['error_2'])-.020
    decision = ('incomplete: nonfinite chemistry metrics' if not all(np.isfinite(m[k]) for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')) else
                'feasible: next step is recompiling the 7-bit angle loaders' if feasible else
                'small gap (<5 mHa): quantisation-aware retraining on GPU' if gap < .005 else
                'large gap: close direction')
    write_summary(dict(run_id=RUN_ID, status=status, seed=SEED, release=RELEASE_DATASET,
        baseline_8bit=BASE_NATIVE, naive=[small_state(s) for s in controls], smart=small_state(state),
        feasible_chemistry=feasible, full_circuit_verified=False,
        refit_ran=refit_ran, refit_lambda_eff=m['lambda_eff'] if refit_ran else None,
        modeled_costs=costs, decision=decision, phase_log=PHASE_LOG,
        elapsed_minutes=(time.monotonic()-NB_T0)/60, best_npz=best_path, candidate_npz=candidate_path))
    print('Checkpoint:', state['label'], '|', decision, flush=True)

def research():
    os.setsid()
    try:
        setup()
        costs = cost_table(BASE_NATIVE['lambda_eff'])
        # A complete baseline is saved before entering any killable search phase.
        fallback = record(WORDS8//2, W8, P8, 'floor_public_W')
        controls = [fallback]
        publish(fallback, controls, costs)
        n = run_phase('naive', naive_search, (), dict(best=fallback, controls=controls))
        best, controls = n['best'], n['controls']; publish(best, controls, costs)
        best = run_phase('progressive', progressive_search, (best,), best); publish(best, controls, costs)
        best = run_phase('polish', polish_search, (best,), best); publish(best, controls, costs)
        best = run_phase('one_body', one_body_search, (best,), best)
        # Revalidate cached one-body checkpoints if that phase hit its hard timeout.
        best = record(best['words'], best['W'], best['p'], 'smart_final')
        pre_refit = small_state(best)
        publish(best, controls, costs)
        refit_ran = (best['metrics']['error_1'] <= REFIT_TRIGGER
                     and best['metrics']['error_2'] <= REFIT_TRIGGER
                     and time.monotonic() < HARD_END-90)
        if refit_ran:
            best = run_phase('refit', constrained_refit, (best,), best)
        else:
            print('Refit skipped: both errors must be <=19.95 mHa and session budget must remain', flush=True)
        best = record(best['words'], best['W'], best['p'], best['label'])
        # Cost projections use the final lambda, while the original table remains inspectable.
        final_costs = cost_table(best['metrics']['lambda_eff'])
        publish(best, controls, final_costs, refit_ran=refit_ran, status='complete')
        summary = json.loads(SUMMARY_PATH.read_text()); summary['smart_before_refit'] = pre_refit
        summary['baseline_cost_model'] = costs; write_summary(summary)
        if v_dfthcsos.passes_accuracy(best['metrics']):
            zpath = ROOT/'rounding_7bit_candidate.npz'
            # Official schema + explicit brot=7 evaluation, without loading angle circuits.
            candidate = v_dfthcsos.load_inputs(zpath)
            checked = v_dfthcsos.evaluate_words(candidate, best['words'], EDGES, 7)
            assert v_dfthcsos.passes_accuracy(checked)
            for k in ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff'):
                assert abs(checked[k]-best['metrics'][k]) < 2e-8
            print('Candidate NPZ passes official numerical accuracy at brot=7', flush=True)
    except Exception as exc:
        traceback.print_exc()
        summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else {}
        summary.update(run_id=RUN_ID, status='failed', failure=repr(exc))
        write_summary(summary)
        raise

# Clear only this notebook's named outputs; never reuse a previous run's verdict.
for path in (SUMMARY_PATH, ROOT/'rounding_7bit_best.npz', ROOT/'rounding_7bit_candidate.npz'):
    path.unlink(missing_ok=True)
ctx = mp.get_context('fork')
process = ctx.Process(target=research)
process.start()
while process.is_alive() and time.monotonic() < HARD_END:
    process.join(timeout=min(20., max(.01, HARD_END-time.monotonic())))
    if process.is_alive():
        print('Session minutes remaining:', round(max(0., HARD_END-time.monotonic())/60, 1), flush=True)
expired = process.is_alive()
if expired:
    # Stop the entire group, including an in-flight phase process, to avoid orphan computation.
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.join(timeout=5)
summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else {}
if expired:
    summary.update(run_id=RUN_ID, status='global timeout; last checkpoint retained',
                   elapsed_minutes=(time.monotonic()-NB_T0)/60)
    write_summary(summary)
print('Research exit:', process.exitcode, '| summary:', SUMMARY_PATH, flush=True)
if not expired and process.exitcode != 0:
    raise RuntimeError('Research aborted; inspect traceback and rounding_7bit.json')


## Verdict

Compare floor, half-up, smart rounding and (when triggered) the constrained refit in the JSON report. Gaps are signed: a positive value exceeds 20 mHa. Feasible chemistry means the official numerical evaluator and `passes_accuracy` passed; loader recompilation and circuit tests are the next experiment. A miss below 5 mHa motivates GPU quantisation-aware retraining; a larger miss closes this direction under the current time budget. An incomplete or timed-out search is explicitly labeled and cannot establish a lower bound on achievable rounding error.


In [ ]:
summary = json.loads(SUMMARY_PATH.read_text())
print('Run status:', summary.get('status'), flush=True)
records = summary.get('naive', [])
if summary.get('smart_before_refit'):
    records = records+[summary['smart_before_refit']]
if summary.get('smart'):
    records = records+[summary['smart']]
if records:
    print(pd.DataFrame([{k:r[k] for k in ('label', 'error_1_mHa', 'error_2_mHa')} for r in records]).to_string(index=False), flush=True)
    print('Gap to 20 mHa:', summary['smart']['gap_to_20_mHa'], flush=True)
    print('Decision:', summary.get('decision'), flush=True)
    if summary.get('status') != 'complete':
        print('Incomplete run: decision is provisional; inspect latest_phase_checkpoint and saved NPZ', flush=True)
        if summary.get('latest_phase_checkpoint'):
            print('Latest saved phase checkpoint:', summary['latest_phase_checkpoint'], flush=True)
    print('Refit lambda_eff:', summary.get('refit_lambda_eff'), flush=True)
    print('Modeled costs/LB (not circuit verified):', flush=True)
    print(pd.DataFrame(summary.get('modeled_costs', [])).to_string(index=False), flush=True)
print('Saved:', SUMMARY_PATH, '|', summary.get('best_npz'), flush=True)


# Sources and credits

- **Lera / marculera** — public compact `(9,143,40)` factorization, angle words and rounding results; the starting chemistry solution is theirs.
- **Dmitrii Gluzdov** — optimized angle loaders and the public Toffoli/qubit cost breakdown (9,492 angle Toffolis, J=5 and 77 temporary qubits).
- **heliosli** — public weights and the authenticated `heliosli/qcc-circuits-live` dataset; **yoyobar** — the supported local portfolio release.
- **Ahmed Berat Özer** — seven-bit savings estimate (about 14%) and antithetic rounding idea; **Phuc Thai** — seven-bit retraining study and the reported naive-rounding failure (about 66 mHa).
- **theredbluepill** — [discussion 753738](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation/discussion/753738), suggesting six- to seven-bit angle words.
- **Tanuj Khattar et al.** — [arXiv:2610.23832](https://arxiv.org/abs/2610.23832), the competition and verifiable scoring framework.
- **Competition verifier** — attached `dfthcsos/verify.py`, `scoreinfra/dfthc_math.py`, `costs_prepare.py`, `circuits.py` and `SCORING_README.md`; source signatures and seven-bit range validation were checked against local verbatim copies when authoring this notebook.
- **Ours** — notebooks 02/04/05 supply the native-parity fixed-word model, packed-pair target, block-coordinate eigen updates and analytic vector gradient; this notebook adds progressive floor/ceil decisions, the analytic Givens reverse pass, exact rank-two flip acceptance and bounded one-body rounding.

Runtime planning on Kaggle's four-core CPU: loading/parity/self-checks about 2–10 minutes; naive controls 5–15 minutes; progressive rounding 20–40 minutes; exact polish 10–20 minutes; one-body search 2–10 minutes; constrained refit 0–30 minutes. These are unmeasured estimates. Phase alarms and the overall 140-minute process watchdog take precedence. No notebook computation was executed locally during authoring.
